# 02 — SINDy identification from simulation data

Simulate the mass-spring-damper, then identify its dynamics with SINDy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from control_lab.models.lti import LTIModel
from control_lab.design.pid import PIDController
from control_lab.sim.backend_control import ControlBackend

%matplotlib inline

## Generate simulation data

In [ ]:
model = LTIModel.mass_spring_damper(m=1.0, k=1.0, c=0.5)

# Open-loop: apply a unit-step force input for identification
from control_lab.design.lqr import LQRController, lqr_continuous

Q = np.eye(2) * 10
R = np.eye(1)
K, _, _ = lqr_continuous(model.A, model.B, Q, R)
ctrl = LQRController(K)

backend = ControlBackend()
r_func = lambda t: np.array([np.sin(2 * np.pi * 0.5 * t), 0.0])
result = backend.simulate(model, ctrl, np.array([0.5, 0.0]), r_func, (0.0, 20.0), 0.01)

data = {'x': result.x, 't': result.t}
print('Data shape:', result.x.shape)

## Fit SINDy model

In [ ]:
from control_lab.ident.sindy_fit import SINDyIdentifier

ident = SINDyIdentifier(dt=0.01)
ident.fit(data)
print('Discovered equations:')
for eq in ident.get_equations():
    print(' ', eq)

## Validate the identified model

In [ ]:
from control_lab.ident.sindy_validate import rollout_error, plot_validation

err = rollout_error(ident, data)
print(f'Rollout RMSE: {err:.4f}')

fig = plot_validation(ident, data, title='SINDy Validation — MSD')
plt.show()